In [1]:
# ============================================================
# 03_autoencoders.py
# Five autoencoder architectures — GPU
# AE | DAE | VAE | Beta-VAE | Batch-VAE
# ============================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import (
    DataLoader, TensorDataset, random_split)
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')
import os
import json

PROJECT_DIR = '/rds/homes/j/jxt554'
DATA_DIR    = f'{PROJECT_DIR}/data'
TABLES_DIR  = f'{PROJECT_DIR}/tables'
SEED        = 42
DEVICE      = torch.device('cuda')
BATCH_SIZE  = 64
EPOCHS      = 300
PATIENCE    = 20
K           = 2

torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True

print('AUTOENCODER TRAINING — GPU')
print('='*55)
print(f'Device : {torch.cuda.get_device_name(0)}')
print(f'Memory : '
      f'{torch.cuda.get_device_properties(0).total_memory//1024**3} GB')

X_cd = np.load(f'{DATA_DIR}/X_cd_int.npy')
X_uc = np.load(f'{DATA_DIR}/X_uc_int.npy')
meta_cd = pd.read_csv(
    f'{DATA_DIR}/meta_cd_sampleqc.csv')
meta_uc = pd.read_csv(
    f'{DATA_DIR}/meta_uc_sampleqc.csv')

print(f'\nCD: {X_cd.shape}')
print(f'UC: {X_uc.shape}')

INPUT_DIM_CD = X_cd.shape[1]
INPUT_DIM_UC = X_uc.shape[1]
N_BATCH      = 2

# Batch one-hot for Batch-VAE
def make_batch_onehot(meta, n_batch=2):
    b = np.zeros((len(meta), n_batch))
    for i, val in enumerate(
            meta['batch'].values):
        b[i, int(val)] = 1.0
    return b

batch_cd = make_batch_onehot(meta_cd)
batch_uc = make_batch_onehot(meta_uc)

print(f'\nCD batch: '
      f'0={(meta_cd["batch"]==0).sum()} '
      f'1={(meta_cd["batch"]==1).sum()}')
print(f'UC batch: '
      f'0={(meta_uc["batch"]==0).sum()} '
      f'1={(meta_uc["batch"]==1).sum()}')

AUTOENCODER TRAINING — GPU
Device : NVIDIA A100-SXM4-40GB
Memory : 39 GB

CD: (215, 991)
UC: (430, 991)

CD batch: 0=189 1=26
UC batch: 0=358 1=72


In [2]:
# ── Model definitions ─────────────────────────────

class AE(nn.Module):
    def __init__(self, input_dim,
                  latent_dim, hidden,
                  dropout):
        super().__init__()
        # Encoder
        enc = []
        prev = input_dim
        for h in hidden:
            enc += [nn.Linear(prev, h),
                    nn.BatchNorm1d(h),
                    nn.ReLU(),
                    nn.Dropout(dropout)]
            prev = h
        enc.append(nn.Linear(prev, latent_dim))
        self.encoder = nn.Sequential(*enc)
        # Decoder
        dec = []
        prev = latent_dim
        for h in reversed(hidden):
            dec += [nn.Linear(prev, h),
                    nn.BatchNorm1d(h),
                    nn.ReLU(),
                    nn.Dropout(dropout)]
            prev = h
        dec.append(nn.Linear(prev, input_dim))
        self.decoder = nn.Sequential(*dec)

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z


class DAE(nn.Module):
    def __init__(self, input_dim,
                  latent_dim, hidden,
                  dropout, noise=0.3):
        super().__init__()
        self.noise = noise
        enc = []
        prev = input_dim
        for h in hidden:
            enc += [nn.Linear(prev, h),
                    nn.BatchNorm1d(h),
                    nn.ReLU(),
                    nn.Dropout(dropout)]
            prev = h
        enc.append(nn.Linear(prev, latent_dim))
        self.encoder = nn.Sequential(*enc)
        dec = []
        prev = latent_dim
        for h in reversed(hidden):
            dec += [nn.Linear(prev, h),
                    nn.BatchNorm1d(h),
                    nn.ReLU(),
                    nn.Dropout(dropout)]
            prev = h
        dec.append(nn.Linear(prev, input_dim))
        self.decoder = nn.Sequential(*dec)

    def forward(self, x, training=True):
        xn = (x + torch.randn_like(x)
               * self.noise
               if training else x)
        z  = self.encoder(xn)
        return self.decoder(z), z


class VAE(nn.Module):
    def __init__(self, input_dim,
                  latent_dim, hidden,
                  dropout):
        super().__init__()
        enc = []
        prev = input_dim
        for h in hidden:
            enc += [nn.Linear(prev, h),
                    nn.BatchNorm1d(h),
                    nn.ReLU(),
                    nn.Dropout(dropout)]
            prev = h
        self.encoder   = nn.Sequential(*enc)
        self.fc_mu     = nn.Linear(
            prev, latent_dim)
        self.fc_logvar = nn.Linear(
            prev, latent_dim)
        dec = []
        prev = latent_dim
        for h in reversed(hidden):
            dec += [nn.Linear(prev, h),
                    nn.BatchNorm1d(h),
                    nn.ReLU(),
                    nn.Dropout(dropout)]
            prev = h
        dec.append(nn.Linear(prev, input_dim))
        self.decoder = nn.Sequential(*dec)

    def reparameterise(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + std * torch.randn_like(std)

    def forward(self, x):
        h      = self.encoder(x)
        mu     = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        z      = self.reparameterise(
            mu, logvar)
        return self.decoder(z), mu, logvar, z


class BatchVAE(nn.Module):
    def __init__(self, input_dim,
                  latent_dim, hidden,
                  dropout, n_batch=2):
        super().__init__()
        enc = []
        prev = input_dim + n_batch
        for h in hidden:
            enc += [nn.Linear(prev, h),
                    nn.BatchNorm1d(h),
                    nn.ReLU(),
                    nn.Dropout(dropout)]
            prev = h
        self.encoder   = nn.Sequential(*enc)
        self.fc_mu     = nn.Linear(
            prev, latent_dim)
        self.fc_logvar = nn.Linear(
            prev, latent_dim)
        dec = []
        prev = latent_dim + n_batch
        for h in reversed(hidden):
            dec += [nn.Linear(prev, h),
                    nn.BatchNorm1d(h),
                    nn.ReLU(),
                    nn.Dropout(dropout)]
            prev = h
        dec.append(nn.Linear(prev, input_dim))
        self.decoder = nn.Sequential(*dec)

    def reparameterise(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + std * torch.randn_like(std)

    def forward(self, x, b):
        h      = self.encoder(
            torch.cat([x, b], dim=1))
        mu     = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        z      = self.reparameterise(
            mu, logvar)
        return (self.decoder(
            torch.cat([z, b], dim=1)),
            mu, logvar, z)


def vae_loss(recon, x, mu, logvar,
              beta=1.0):
    r  = nn.functional.mse_loss(
        recon, x, reduction='sum')
    kl = -0.5 * torch.sum(
        1 + logvar - mu.pow(2)
        - logvar.exp())
    return r + beta * kl

print('Model definitions loaded ✓')

Model definitions loaded ✓


In [3]:
# ── Training function ─────────────────────────────
def train_model(X, batch_oh=None,
                 input_dim=None,
                 latent_dim=32,
                 hidden=[256,128,64],
                 dropout=0.3,
                 lr=0.001,
                 model_type='ae',
                 beta=1.0):

    X_t = torch.FloatTensor(X).to(DEVICE)
    if batch_oh is not None:
        B_t = torch.FloatTensor(
            batch_oh).to(DEVICE)

    n_val = int(len(X_t) * 0.2)
    n_tr  = len(X_t) - n_val

    if model_type == 'batch_vae':
        ds = TensorDataset(X_t, B_t)
    else:
        ds = TensorDataset(X_t)

    tr_ds, val_ds = random_split(
        ds, [n_tr, n_val],
        generator=torch.Generator().manual_seed(SEED))
    tr_dl  = DataLoader(
        tr_ds, batch_size=BATCH_SIZE,
        shuffle=True)
    val_dl = DataLoader(
        val_ds, batch_size=BATCH_SIZE)

    # Build model
    if model_type == 'ae':
        model = AE(input_dim, latent_dim,
                    hidden, dropout).to(DEVICE)
    elif model_type == 'dae':
        model = DAE(input_dim, latent_dim,
                     hidden, dropout).to(DEVICE)
    elif model_type in ('vae','beta_vae'):
        model = VAE(input_dim, latent_dim,
                     hidden, dropout).to(DEVICE)
    elif model_type == 'batch_vae':
        model = BatchVAE(
            input_dim, latent_dim,
            hidden, dropout, N_BATCH).to(DEVICE)

    opt  = torch.optim.Adam(
        model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    best_val   = float('inf')
    patience_  = 0
    best_state = None

    for ep in range(EPOCHS):
        model.train()
        for batch in tr_dl:
            opt.zero_grad()
            if model_type == 'ae':
                xb      = batch[0]
                recon,_ = model(xb)
                loss    = loss_fn(recon, xb)
            elif model_type == 'dae':
                xb      = batch[0]
                recon,_ = model(xb, True)
                loss    = loss_fn(recon, xb)
            elif model_type in (
                    'vae','beta_vae'):
                xb = batch[0]
                recon,mu,lv,_ = model(xb)
                loss = vae_loss(
                    recon,xb,mu,lv,beta)
            elif model_type == 'batch_vae':
                xb, bb = batch
                recon,mu,lv,_ = model(xb,bb)
                loss = vae_loss(
                    recon,xb,mu,lv,beta)
            loss.backward()
            opt.step()

        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_dl:
                if model_type == 'ae':
                    xb      = batch[0]
                    recon,_ = model(xb)
                    val_loss += loss_fn(
                        recon,xb).item()
                elif model_type == 'dae':
                    xb      = batch[0]
                    recon,_ = model(xb,False)
                    val_loss += loss_fn(
                        recon,xb).item()
                elif model_type in (
                        'vae','beta_vae'):
                    xb = batch[0]
                    recon,mu,lv,_ = model(xb)
                    val_loss += vae_loss(
                        recon,xb,mu,lv,
                        beta).item()
                elif model_type == 'batch_vae':
                    xb, bb = batch
                    recon,mu,lv,_ = model(
                        xb,bb)
                    val_loss += vae_loss(
                        recon,xb,mu,lv,
                        beta).item()
        val_loss /= len(val_dl)

        if val_loss < best_val:
            best_val   = val_loss
            patience_  = 0
            best_state = {
                k: v.clone()
                for k,v in
                model.state_dict().items()}
        else:
            patience_ += 1
            if patience_ >= PATIENCE:
                break

    model.load_state_dict(best_state)
    model.eval()

    # Extract latent
    with torch.no_grad():
        if model_type == 'ae':
            _,z = model(X_t)
        elif model_type == 'dae':
            _,z = model(X_t, False)
        elif model_type in ('vae','beta_vae'):
            _,mu,_,_ = model(X_t)
            z = mu
        elif model_type == 'batch_vae':
            _,mu,_,_ = model(X_t, B_t)
            z = mu

    return best_val, z.cpu().numpy()

print('Training function defined ✓')

Training function defined ✓


In [4]:
# ── Grid search ───────────────────────────────────
LATENT_DIMS = [8, 16, 32, 64, 128]
HIDDEN_DIMS = [[256,128,64],
               [512,256,128]]
DROPOUTS    = [0.2, 0.3]
LRS         = [0.001, 0.0005]
BETAS       = [1.0, 2.0, 4.0]

MODEL_CONFIGS = [
    ('AE',        'ae',        [1.0]),
    ('DAE',       'dae',       [1.0]),
    ('VAE',       'vae',       [1.0]),
    ('Beta-VAE',  'beta_vae',  BETAS),
    ('Batch-VAE', 'batch_vae', [1.0, 2.0]),
]

def run_grid_search(X, batch_oh,
                     input_dim, label):
    print(f'\n{"="*55}')
    print(f'{label} GRID SEARCH')
    print(f'{"="*55}')

    all_results  = []
    best_per_model = {}

    for model_name, model_type, betas in \
            MODEL_CONFIGS:
        print(f'\n--- {model_name} ---')
        best_sil    = -1
        best_latent = None
        best_ld     = None
        best_cfg    = None

        for ld in LATENT_DIMS:
            for hd in HIDDEN_DIMS:
                for do in DROPOUTS:
                    for lr in LRS:
                        for beta in betas:
                            try:
                                val_loss, latent = \
                                    train_model(
                                        X,
                                        batch_oh
                                        if model_type=='batch_vae'
                                        else None,
                                        input_dim,
                                        ld, hd,
                                        do, lr,
                                        model_type,
                                        beta)
                                km  = KMeans(
                                    n_clusters=K,
                                    random_state=SEED,
                                    n_init=20)
                                lbl = km.fit_predict(
                                    latent)
                                if len(np.unique(
                                        lbl)) < 2:
                                    continue
                                sil = silhouette_score(
                                    latent, lbl)
                                all_results.append({
                                    'cohort'    : label,
                                    'model'     : model_name,
                                    'latent_dim': ld,
                                    'hidden'    : str(hd),
                                    'dropout'   : do,
                                    'lr'        : lr,
                                    'beta'      : beta,
                                    'val_loss'  : val_loss,
                                    'silhouette': sil})
                                if sil > best_sil:
                                    best_sil    = sil
                                    best_latent = latent.copy()
                                    best_ld     = ld
                                    best_cfg    = {
                                        'latent_dim': ld,
                                        'hidden'    : hd,
                                        'dropout'   : do,
                                        'lr'        : lr,
                                        'beta'      : beta}
                            except Exception as e:
                                pass

        best_per_model[model_name] = {
            'latent': best_latent,
            'sil'   : best_sil,
            'ld'    : best_ld,
            'cfg'   : best_cfg}

        fname = (model_name.lower()
                  .replace('-','_')
                  .replace(' ','_'))
        if best_latent is not None:
            np.save(
                f'{DATA_DIR}/'
                f'latent_{label.lower()}'
                f'_{fname}_best.npy',
                best_latent)
            with open(
                f'{DATA_DIR}/'
                f'cfg_{label.lower()}'
                f'_{fname}.json','w') as f:
                json.dump(best_cfg, f,
                           indent=2)

        print(f'  Best: ld={best_ld} '
              f'sil={best_sil:.4f}')
        if best_cfg:
            print(f'  Config: {best_cfg}')

    # Save all results
    pd.DataFrame(all_results).to_csv(
        f'{TABLES_DIR}/'
        f'ae_grid_{label.lower()}.csv',
        index=False)

    print(f'\n{label} SUMMARY:')
    print(f'{"Model":<12} {"Best ld":>8} '
          f'{"Best sil":>10}')
    print('-'*32)
    for mn, res in best_per_model.items():
        print(f'{mn:<12} {res["ld"]:>8} '
              f'{res["sil"]:>10.4f}')

    return best_per_model, all_results

# Run CD
cd_ae_results, cd_ae_all = run_grid_search(
    X_cd, batch_cd, INPUT_DIM_CD, 'CD')

# Run UC
uc_ae_results, uc_ae_all = run_grid_search(
    X_uc, batch_uc, INPUT_DIM_UC, 'UC')

print('\nAUTOENCODER GRID SEARCH COMPLETE')
print('Next: 04_dec.py')


CD GRID SEARCH

--- AE ---
  Best: ld=128 sil=0.4352
  Config: {'latent_dim': 128, 'hidden': [256, 128, 64], 'dropout': 0.3, 'lr': 0.0005, 'beta': 1.0}

--- DAE ---
  Best: ld=64 sil=0.4897
  Config: {'latent_dim': 64, 'hidden': [256, 128, 64], 'dropout': 0.3, 'lr': 0.0005, 'beta': 1.0}

--- VAE ---
  Best: ld=16 sil=0.3461
  Config: {'latent_dim': 16, 'hidden': [256, 128, 64], 'dropout': 0.3, 'lr': 0.001, 'beta': 1.0}

--- Beta-VAE ---
  Best: ld=32 sil=0.3825
  Config: {'latent_dim': 32, 'hidden': [256, 128, 64], 'dropout': 0.3, 'lr': 0.001, 'beta': 4.0}

--- Batch-VAE ---
  Best: ld=16 sil=0.3731
  Config: {'latent_dim': 16, 'hidden': [256, 128, 64], 'dropout': 0.3, 'lr': 0.001, 'beta': 2.0}

CD SUMMARY:
Model         Best ld   Best sil
--------------------------------
AE                128     0.4352
DAE                64     0.4897
VAE                16     0.3461
Beta-VAE           32     0.3825
Batch-VAE          16     0.3731

UC GRID SEARCH

--- AE ---
  Best: ld=128 sil=0.48